In [1]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import json
import sklearn
import numpy as np

# Reading the graph

In [2]:
df = pd.read_csv("lasftm_asia/lastfm_asia_edges.csv", dtype={"node_1": str, "node_2": str})

G = nx.from_pandas_edgelist(df, source="node_1", target="node_2")

with open("lasftm_asia/lastfm_asia_features.json", "r") as f:
    features = json.load(f)

for node, feat in features.items():
    if node in G:
        G.nodes[node]["feat"] = feat

df = pd.read_csv("lasftm_asia/lastfm_asia_target.csv", dtype={"id": str})

target_map = dict(zip(df["id"], df["target"]))

nx.set_node_attributes(G, target_map, "target")


In [3]:
def one_hot(i, dim):
    vec = np.zeros(dim, dtype=np.float32)
    vec[i] = 1.0
    return vec

In [4]:
nodes = list(G.nodes())
node_to_idx = {n: i for i, n in enumerate(nodes)}

feat_dim = max(
    max(v) for v in features.values() if len(v) > 0
) + 1

nodes = list(G.nodes())

X = np.array([
    one_hot(features[n], feat_dim)
    for n in nodes
], dtype=np.float32)

y = np.array([
    list(target_map.values())
], dtype=np.int64)

edge_index = []

for u, v in G.edges():
    u_i = node_to_idx[u]
    v_i = node_to_idx[v]

    edge_index.append([u_i, v_i])
    edge_index.append([v_i, u_i])  # undirected

edge_index = np.array(edge_index).T

In [5]:
num_nodes = len(nodes)
perm = np.random.permutation(num_nodes)

train_idx = perm[:int(0.6 * num_nodes)]
val_idx   = perm[int(0.6 * num_nodes):int(0.8 * num_nodes)]
test_idx  = perm[int(0.8 * num_nodes):]

In [6]:
from torch_geometric.data import Data
import torch

data = Data(
    x=torch.tensor(X),
    edge_index=torch.tensor(edge_index, dtype=torch.long),
    y=torch.tensor(y)
)
data.y = data.y.view(-1)

ModuleNotFoundError: No module named 'torch_geometric'

In [ ]:
data.train_mask = torch.zeros(num_nodes, dtype=torch.bool)
data.val_mask   = torch.zeros(num_nodes, dtype=torch.bool)
data.test_mask  = torch.zeros(num_nodes, dtype=torch.bool)

data.train_mask[train_idx] = True
data.val_mask[val_idx] = True
data.test_mask[test_idx] = True

# Model definition

In [ ]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv

class GCN(torch.nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, num_layers=2, dropout=0.5):
        super().__init__()

        assert num_layers >= 2, "Need at least 2 layers"

        self.num_layers = num_layers
        self.dropout = dropout

        # input layer
        self.convs = torch.nn.ModuleList()
        self.convs.append(GCNConv(in_dim, hidden_dim))

        # hidden layers
        for _ in range(num_layers - 2):
            self.convs.append(GCNConv(hidden_dim, hidden_dim))

        # output layer
        self.convs.append(GCNConv(hidden_dim, out_dim))

    def forward(self, data):
        x, edge_index = data.x, data.edge_index

        for i, conv in enumerate(self.convs):
            x = conv(x, edge_index)

            # no activation on last layer
            if i != len(self.convs) - 1:
                x = F.relu(x)
                x = F.dropout(x, p=self.dropout, training=self.training)

        return x

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = GCN(
    in_dim=data.x.shape[1],
    hidden_dim=1024,
    num_layers=5,
    dropout=0.0,
    out_dim=len(np.unique(y))
)

# counts = torch.bincount(data.y[data.val_mask], minlength=len(np.unique(y)))
# weights = 1.0 / counts.float()

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn = torch.nn.CrossEntropyLoss(weight=None)

model = model.to(device)
data = data.to(device)

In [ ]:
import torch
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report

train_accs = []
val_accs = []
losses = []

for epoch in range(200):
    model.train()
    optimizer.zero_grad()

    out = model(data)
    loss = loss_fn(out[data.train_mask], data.y[data.train_mask])

    loss.backward()
    optimizer.step()

    # evaluation
    model.eval()
    with torch.no_grad():
        pred = out.argmax(dim=1)

        train_acc = (pred[data.train_mask] == data.y[data.train_mask]).float().mean()
        val_acc = (pred[data.val_mask] == data.y[data.val_mask]).float().mean()

    losses.append(loss.item())
    train_accs.append(train_acc.item())
    val_accs.append(val_acc.item())

    if epoch % 10 == 0:
        print(f"Epoch {epoch} | Loss {loss.item():.4f} | Train {train_acc:.4f} | Val {val_acc:.4f}")

Epoch 0 | Loss 2.8816 | Train 0.0645 | Val 0.0662
Epoch 10 | Loss 2.4499 | Train 0.1670 | Val 0.1816
Epoch 20 | Loss 2.4271 | Train 0.2070 | Val 0.2039
Epoch 30 | Loss 2.4129 | Train 0.2075 | Val 0.2039
Epoch 40 | Loss 2.3817 | Train 0.2134 | Val 0.1830
Epoch 50 | Loss 2.3377 | Train 0.2335 | Val 0.1869
Epoch 60 | Loss 2.3138 | Train 0.2140 | Val 0.1607
Epoch 70 | Loss 2.2646 | Train 0.2634 | Val 0.1725
Epoch 80 | Loss 2.2251 | Train 0.2825 | Val 0.1889
Epoch 90 | Loss 2.1663 | Train 0.2803 | Val 0.1607
Epoch 100 | Loss 2.1957 | Train 0.2687 | Val 0.1134
Epoch 110 | Loss 2.0657 | Train 0.3144 | Val 0.1816
Epoch 120 | Loss 1.9711 | Train 0.3400 | Val 0.1574
Epoch 130 | Loss 2.1491 | Train 0.3091 | Val 0.1495
Epoch 140 | Loss 1.9848 | Train 0.3400 | Val 0.1587
Epoch 150 | Loss 1.8976 | Train 0.3603 | Val 0.1528
Epoch 160 | Loss 1.9785 | Train 0.3537 | Val 0.1843
Epoch 170 | Loss 1.8407 | Train 0.3841 | Val 0.1600
Epoch 180 | Loss 1.8372 | Train 0.3824 | Val 0.1338
Epoch 190 | Loss 1.6989

In [ ]:
print(out.shape)
print(data.y.shape)
print(data.train_mask.shape)

torch.Size([7624, 18])
torch.Size([1, 7624])
torch.Size([7624])
